In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Consolidated block

In [ ]:
import os
import sys
import copy
import math
import random
import time
import warnings
import json
import hashlib
import inspect
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, roc_auc_score
from sklearn.preprocessing import label_binarize
from imblearn.over_sampling import RandomOverSampler
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warning outputs to maintain clean terminal execution logs
warnings.filterwarnings('ignore')

# =====================================================================
# 1. DYNAMIC PATH RESOLUTION WITH COLAB DRIVE SUPPORT
# =====================================================================
# This block dynamically searches for the network attack dataset 'data.csv'
# across active directories and mounted Google Drive volumes.
def find_data_file(filename="data.csv"):
    search_paths = [
        "/content/gdrive/MyDrive",
        "/content/gdrive/My Drive",
        "/content/drive/MyDrive",
        "/content/drive/My Drive",
        "/content/gdrive",
        "/content/drive",
        "/content"
    ]
    for path in search_paths:
        full_path = os.path.join(path, filename)
        if os.path.exists(full_path):
            return full_path
    for root, dirs, files in os.walk("/content"):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"Could not locate '{filename}' in the notebook directory hierarchy.")

try:
    DATA_PATH = find_data_file("data.csv")
    print(f"Resolved dataset location to: {DATA_PATH}")
except Exception as e:
    print(f"Warning: {e}. Defaulting to standard fallback path.")
    DATA_PATH = "/content/gdrive/MyDrive/data.csv"

# =====================================================================
# 2. SEED SELECTION AND SYSTEM HYPERPARAMETERS
# =====================================================================
# Core system settings to guarantee total determinism, reproducibility,
# and specific partitions across the PATE pipeline.
SEED = 42
TARGET_COLUMN = "is_attack"
SELECTED_CLASSES = [0, 1]

BATCH_SIZE = 128
TEACHER_EPOCHS = 23
STUDENT_EPOCHS = 30
LEARNING_RATE = 1e-4
PATIENCE = 3

TEACHER_VALID_FRACTION = 0.30
STUDENT_VALID_FRACTION = 0.20
TEACHER_POOL_FRACTION = 0.80
STUDENT_PRIVATE_FRACTION = 0.00
PUBLIC_FRACTION = 0.05
FINAL_TEST_FRACTION = 0.15

USE_OVERSAMPLING = True
LAPLACE_INVERSE_SCALE = 1.3
DELTA = 1e-5
MAX_MOMENT = 32
OUTPUT_DIR = "./pate_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# 3. DATASET LOADING AND PIPELINE SPLITTING
# =====================================================================
# Loads the target columns, applies class restrictions, drops missing/
# non-numeric elements, and splits the data into teacher, public, and test folds.
def load_dataset(path, target_column, selected_classes):
    data = pd.read_csv(path, index_col=0)
    if target_column not in data.columns:
        raise ValueError(f"Target column '{target_column}' was not found.")
    data = data[data[target_column].isin(selected_classes)].copy()
    if len(data) == 0:
        raise ValueError("No observations remain after class filtering.")
    return data

def preprocess_and_split(data, target_column, seed=42):
    data = data.copy()
    total_fraction = TEACHER_POOL_FRACTION + STUDENT_PRIVATE_FRACTION + PUBLIC_FRACTION + FINAL_TEST_FRACTION
    if not np.isclose(total_fraction, 1.0):
        raise ValueError("Dataset fractions must sum to 1.0.")

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(data[target_column])
    X = data.drop(columns=[target_column])

    non_numeric = X.select_dtypes(include=["object", "category"]).columns.tolist()
    if len(non_numeric) > 0:
        X = X.drop(columns=non_numeric)

    X = X.apply(pd.to_numeric, errors="coerce")
    valid_rows = ~X.isna().any(axis=1)
    X = X.loc[valid_rows].reset_index(drop=True)
    y = y[valid_rows.to_numpy()]

    X = X.to_numpy(dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    X_teacher, X_heldout, y_teacher, y_heldout = train_test_split(
        X, y, test_size=(1.0 - TEACHER_POOL_FRACTION), random_state=seed, stratify=y
    )

    heldout_total_sub_fraction = STUDENT_PRIVATE_FRACTION + PUBLIC_FRACTION + FINAL_TEST_FRACTION
    test_size_final_test = FINAL_TEST_FRACTION / heldout_total_sub_fraction

    X_public, X_final_test, y_public, y_final_test = train_test_split(
        X_heldout, y_heldout, test_size=test_size_final_test, random_state=seed, stratify=y_heldout
    )

    X_private = np.zeros((0, X_teacher.shape[1]), dtype=np.float32)
    y_private = np.zeros((0,), dtype=np.int64)

    return X_teacher, y_teacher, X_private, y_private, X_public, y_public, X_final_test, y_final_test, label_encoder

class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

# =====================================================================
# 4. CORE ENSEMBLE TEACHER ARCHITECTURES
# =====================================================================
# Specific model definitions for the PATE ensemble teachers, allowing sequence
# processing models (LSTM, GRU), spatial models (CNN), and dense models (MLP).
class TeacherLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, (hidden, _) = self.lstm(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class TeacherGRU(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, hidden = self.gru(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class TeacherCNN(nn.Module):
    def __init__(self, input_size, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * input_size, 128)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)

class TeacherMLP(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=3):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.20))
            current_size = hidden_size
        self.network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 3:
            x = x.mean(dim=1)
        return F.log_softmax(self.output_layer(self.network(x)), dim=1)

def build_teacher(architecture, input_size, num_classes):
    if architecture == "LSTM":
         return TeacherLSTM(input_size=input_size, num_classes=num_classes)
    if architecture == "GRU":
         return TeacherGRU(input_size=input_size, num_classes=num_classes)
    if architecture == "CNN":
        return TeacherCNN(input_size=input_size, num_classes=num_classes)
    if architecture == "MLP":
        return TeacherMLP(input_size=input_size, num_classes=num_classes)
    raise ValueError(f"Unknown teacher architecture: {architecture}")

# =====================================================================
# 5. CORE STUDENT ARCHITECTURES
# =====================================================================
# Architectures matching the structural specs for PATE knowledge distillation.
class StudentLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, (hidden, _) = self.lstm(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class StudentGRU(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, hidden = self.gru(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class StudentCNN(nn.Module):
    def __init__(self, input_size, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * input_size, 128)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)

class StudentMLP(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=3):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.20))
            current_size = hidden_size
        self.network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 3:
            x = x.mean(dim=1)
        return F.log_softmax(self.output_layer(self.network(x)), dim=1)

def build_student(architecture, input_size, num_classes):
    if architecture == "MLP":
        return StudentMLP(input_size=input_size, num_classes=num_classes)
    elif architecture == "LSTM":
        return StudentLSTM(input_size=input_size, num_classes=num_classes)
    elif architecture == "GRU":
        return StudentGRU(input_size=input_size, num_classes=num_classes)
    elif architecture == "CNN":
        return StudentCNN(input_size=input_size, num_classes=num_classes)
    raise ValueError(f"Unknown student architecture: {architecture}")

# =====================================================================
# 6. TRAINING AND OPTIMIZATION ROUTINES
# =====================================================================
# Optimized training framework utilizing early stopping validation metrics.
def train_model(model, train_loader, valid_loader, epochs, learning_rate, patience, device):
    model = model.to(device)
    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    best_validation_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        total_train_loss, total_train, correct_train = 0.0, 0, 0
        for is_input, labels in train_loader:
            is_input, labels = is_input.to(device), labels.to(device)
            optimizer.zero_grad()
            log_probabilities = model(is_input)
            loss = criterion(log_probabilities, labels)
            loss.backward()
            optimizer.step()
            b_size = labels.size(0)
            total_train_loss += loss.item() * b_size
            predictions = torch.argmax(log_probabilities, dim=1)
            correct_train += (predictions == labels).sum().item()
            total_train += b_size

        model.eval()
        total_valid_loss, total_valid, correct_valid = 0.0, 0, 0
        with torch.no_grad():
            for is_input, labels in valid_loader:
                is_input, labels = is_input.to(device), labels.to(device)
                log_probabilities = model(is_input)
                loss = criterion(log_probabilities, labels)
                b_size = labels.size(0)
                total_valid_loss += loss.item() * b_size
                predictions = torch.argmax(log_probabilities, dim=1)
                correct_valid += (predictions == labels).sum().item()
                total_valid += b_size

        valid_loss = total_valid_loss / total_valid
        history.append({"epoch": epoch + 1, "valid_loss": valid_loss})

        if valid_loss < best_validation_loss:
            best_validation_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

def predict_labels(model, loader, device):
    model.eval()
    predictions = []
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = model(inputs)
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            predictions.extend(batch_predictions.cpu().numpy())
    return np.asarray(predictions, dtype=np.int64)

# =====================================================================
# 7. RIGOROUS EVALUATION SYSTEM
# =====================================================================
# Rigorous test evaluation suite calculating Accuracy, Recall, Precision,
# F1-Macro, MAE, and multiclass OVR ROC-AUC scores.
def evaluate_student_rigorous(student_model, X_test, y_test, num_classes, batch_size, device):
    student_model.eval()
    loader = DataLoader(TabularDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
    all_predictions = []
    all_probs = []

    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = student_model(inputs)
            probs = torch.exp(log_probabilities)
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            all_predictions.extend(batch_predictions.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_predictions = np.array(all_predictions)
    all_probs = np.array(all_probs, dtype=np.float64)
    y_test = np.array(y_test, dtype=np.float64)

    accuracy = accuracy_score(y_test, all_predictions)
    precision = precision_score(y_test, all_predictions, average='macro', zero_division=0)
    recall = recall_score(y_test, all_predictions, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, all_predictions, average='macro', zero_division=0)

    if num_classes == 2:
        target_probs = all_probs[:, 1]
        mae = float(np.mean(np.abs(y_test - target_probs)))
        try:
            auc = float(roc_auc_score(y_test, target_probs))
        except Exception:
            auc = np.nan
    else:
        y_test_bin = label_binarize(y_test, classes=np.arange(num_classes)).astype(np.float64)
        mae = float(mean_absolute_error(y_test_bin, all_probs))
        try:
            auc = float(roc_auc_score(y_test_bin, all_probs, average='macro', multi_class='ovr'))
        except Exception:
            auc = np.nan

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1_macro),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "mae": float(mae)
    }

# =====================================================================
# 8. PATE VOTE AGGREGATION AND RDP COMPLIANCE
# =====================================================================
# Implements privacy accounting via Rényi Differential Privacy (RDP).
# Features the safety invariant check to verify dep_eps <= ind_eps.
def teacher_predictions_to_counts(teacher_preds, num_classes):
    teacher_preds = np.asarray(teacher_preds, dtype=np.int64)
    num_teachers, num_queries = teacher_preds.shape
    counts = np.zeros((num_queries, num_classes), dtype=np.int64)
    for query_index in range(num_queries):
        counts[query_index] = np.bincount(teacher_preds[:, query_index], minlength=num_classes)
    return counts

def aggregate_teacher_votes_laplace(teacher_preds, num_classes, laplace_inverse_scale, random_seed=None):
    counts = teacher_predictions_to_counts(teacher_preds, num_classes)
    rng = np.random.default_rng(random_seed)
    laplace_scale = 1.0 / laplace_inverse_scale
    noise = rng.laplace(loc=0.0, scale=laplace_scale, size=counts.shape)
    noisy_counts = counts.astype(np.float64) + noise
    released_labels = np.argmax(noisy_counts, axis=1)
    return released_labels, counts, noisy_counts

def logaddexp_scalar(a, b):
    if a == -math.inf: return b
    if b == -math.inf: return a
    maximum = max(a, b)
    return maximum + math.log(math.exp(a - maximum) + math.exp(b - maximum))

def logmgf_exact(q, mechanism_eps, moment_order):
    lam = float(moment_order)
    eps = float(mechanism_eps)
    bound_1 = 0.5 * (eps ** 2) * lam * (lam + 1.0)
    bound_3 = eps * lam
    bound_2 = math.inf
    denominator = 1.0 - math.exp(eps) * q
    if q < 0.5 and denominator > 0.0:
        log_one_minus_q = math.log1p(-q)
        log_term_1 = log_one_minus_q + lam * (log_one_minus_q - math.log(denominator))
        log_term_2 = math.log(q) + eps * lam if q > 0.0 else -math.inf
        bound_2 = logaddexp_scalar(log_term_1, log_term_2)
    return min(bound_1, bound_2, bound_3)

def compute_q_noisy_max(counts, laplace_inverse_scale):
    counts = np.asarray(counts, dtype=np.float64)
    num_classes = len(counts)
    if num_classes < 2:
        return 0.0
    winner = int(np.argmax(counts))
    winning_votes = counts[winner]
    competitors = np.delete(counts, winner)
    gaps = laplace_inverse_scale * (winning_votes - competitors)
    q = float(np.sum((gaps + 2.0) * np.exp(-gaps) / 4.0))
    return min(max(q, 0.0), 1.0 - 1.0 / num_classes)

def data_dependent_log_moment(counts, laplace_inverse_scale, moment_order):
    q = compute_q_noisy_max(counts, laplace_inverse_scale)
    mechanism_eps = 2.0 * laplace_inverse_scale
    return logmgf_exact(q, mechanism_eps, moment_order)

def data_independent_log_moment(laplace_inverse_scale, moment_order):
    mechanism_eps = 2.0 * laplace_inverse_scale
    lam = float(moment_order)
    concentration_bound = 0.5 * (mechanism_eps ** 2) * lam * (lam + 1.0)
    direct_dp_bound = mechanism_eps * lam
    return min(concentration_bound, direct_dp_bound)

def calculate_pate_privacy(vote_counts, laplace_inverse_scale=1.3, delta=1e-5, moments=32):
    vote_counts = np.asarray(vote_counts, dtype=np.int64)
    num_queries, classes_dim = vote_counts.shape
    orders = np.arange(1, moments + 1, dtype=np.float64)
    dependent_total = np.zeros(moments, dtype=np.float64)
    independent_total = np.zeros(moments, dtype=np.float64)
    q_values, vote_margins = [], []

    for counts in vote_counts:
        q = compute_q_noisy_max(counts, laplace_inverse_scale)
        q_values.append(q)
        sorted_counts = np.sort(counts)
        vote_margins.append(sorted_counts[-1] - sorted_counts[-2])
        for pos, order in enumerate(orders):
            dependent_total[pos] += data_dependent_log_moment(counts, laplace_inverse_scale, order)
            independent_total[pos] += data_independent_log_moment(laplace_inverse_scale, order)

    dep_candidates = (dependent_total - math.log(delta)) / orders
    ind_candidates = (independent_total - math.log(delta)) / orders
    dep_idx = np.argmin(dep_candidates)
    ind_idx = np.argmin(ind_candidates)

    dep_epsilon = float(dep_candidates[dep_idx])
    ind_epsilon = float(ind_candidates[ind_idx])

    # Mathematical safety constraint assert validation
    if dep_epsilon > ind_epsilon + 1e-10:
        raise AssertionError(
            f"Invalid privacy result: dependent epsilon "
            f"{dep_epsilon} exceeds independent epsilon {ind_epsilon}"
        )

    return {
        "data_dependent_epsilon": dep_epsilon,
        "data_independent_epsilon": ind_epsilon,
        "q_values": np.asarray(q_values),
        "vote_margins": np.asarray(vote_margins)
    }

# =====================================================================
# 9. EXPERIMENTAL UNIT TESTING SUITE
# =====================================================================
# Validates mathematical bounds and verification scenarios for the accountant.
def test_dd_never_exceeds_di():
    test_votes = np.array([
        [9, 0, 0],
        [6, 2, 1],
        [4, 3, 2],
        [3, 3, 3]
    ])
    result = calculate_pate_privacy(
        test_votes, laplace_inverse_scale=1.3, delta=1e-5, moments=32
    )
    assert (
        result["data_dependent_epsilon"] <= result["data_independent_epsilon"] + 1e-10
    ), result
    print("PASS: test_dd_never_exceeds_di")

def run_unit_tests():
    print("Starting PATE compliance testing...")
    test_dd_never_exceeds_di()
    print("All system unit tests completed successfully!")

# Run validation tests prior to main experimental block
run_unit_tests()

# =====================================================================
# 10. GENERAL LIFE CYCLE SYSTEM EXPERIMENT PIPELINE
# =====================================================================
# Baseline framework to run a specific end-to-end PATE experiment configuration.
# Localized state-resets are used to simulate isolated executions.
def run_custom_pate_experiment(config):
    # Simulate fresh isolated Colab runtime parameters locally via seed resetting
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    data = load_dataset(DATA_PATH, TARGET_COLUMN, SELECTED_CLASSES)
    X_teacher, y_teacher, X_private, y_private, X_public, y_public, X_final_test, y_final_test, label_encoder = preprocess_and_split(data, TARGET_COLUMN, seed=SEED)

    input_size = X_teacher.shape[1]
    num_classes = len(np.unique(y_teacher))

    scaler = StandardScaler()
    X_teacher_scaled = scaler.fit_transform(X_teacher)
    X_public_scaled = scaler.transform(X_public)
    X_final_test_scaled = scaler.transform(X_final_test)

    n = len(X_teacher_scaled)
    indices = np.arange(n)
    rng = np.random.default_rng(SEED)
    rng.shuffle(indices)
    partitions = np.array_split(indices, config["num_teachers"])

    train_loaders, valid_loaders = [], []
    for teacher_id, teacher_indices in enumerate(partitions):
        X_part = X_teacher_scaled[teacher_indices]
        y_part = y_teacher[teacher_indices]
        X_train, X_valid, y_train, y_valid = train_test_split(X_part, y_part, test_size=TEACHER_VALID_FRACTION, random_state=SEED + teacher_id, stratify=y_part)
        if USE_OVERSAMPLING:
            sampler = RandomOverSampler(random_state=SEED + teacher_id)
            X_train, y_train = sampler.fit_resample(X_train, y_train)
        train_loaders.append(DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True))
        valid_loaders.append(DataLoader(TabularDataset(X_valid, y_valid), batch_size=BATCH_SIZE, shuffle=False))

    teachers = []
    for idx in range(config["num_teachers"]):
        arch = config["teacher_archs"][idx % len(config["teacher_archs"])]
        model = build_teacher(arch, input_size, num_classes)
        trained_model, _ = train_model(model, train_loaders[idx], valid_loaders[idx], TEACHER_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)
        teachers.append(trained_model)

    public_loader = DataLoader(TabularDataset(X_public_scaled, y_public), batch_size=BATCH_SIZE, shuffle=False)
    all_teacher_preds = []
    for idx, t_model in enumerate(teachers):
        preds = predict_labels(t_model, public_loader, DEVICE)
        all_teacher_preds.append(preds)
    all_teacher_preds = np.stack(all_teacher_preds)

    released_labels, vote_counts, _ = aggregate_teacher_votes_laplace(all_teacher_preds, num_classes, LAPLACE_INVERSE_SCALE, random_seed=SEED)
    privacy_results = calculate_pate_privacy(vote_counts, LAPLACE_INVERSE_SCALE, DELTA, MAX_MOMENT)

    mean_vote_margin = float(np.mean(privacy_results["vote_margins"]))

    X_student_train, X_student_valid, y_student_train, y_student_valid = train_test_split(X_public_scaled, released_labels, test_size=STUDENT_VALID_FRACTION, random_state=SEED, stratify=released_labels)
    student_train_loader = DataLoader(TabularDataset(X_student_train, y_student_train), batch_size=BATCH_SIZE, shuffle=True)
    student_valid_loader = DataLoader(TabularDataset(X_student_valid, y_student_valid), batch_size=BATCH_SIZE, shuffle=False)

    student_model = build_student(config["student_arch"], input_size, num_classes)
    trained_student, _ = train_model(student_model, student_train_loader, student_valid_loader, STUDENT_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)

    student_metrics = evaluate_student_rigorous(trained_student, X_final_test_scaled, y_final_test, num_classes, BATCH_SIZE, DEVICE)

    return {
        "dep_eps": privacy_results["data_dependent_epsilon"],
        "ind_eps": privacy_results["data_independent_epsilon"],
        "mean_vote_margin": mean_vote_margin,
        **student_metrics
    }

# =====================================================================
# 11. STRICT 15 CONFIGURATION COMPARISON BENCHMARK
# =====================================================================
# Specifically structured to match the user's requested comparison suite
# with absolutely no missing configurations, repetitions, or exclusions.
all_15_configurations = [
    {
        "name": "1 LSTM teacher 1 GRU student",
        "num_teachers": 1,
        "teacher_archs": ["LSTM"],
        "student_arch": "GRU"
    },
    {
        "name": "3 LSTM teachers 3 CNN teachers 3 MLP teachers 1 LSTM student",
        "num_teachers": 9,
        "teacher_archs": ["LSTM"]*3 + ["CNN"]*3 + ["MLP"]*3,
        "student_arch": "LSTM"
    },
    {
        "name": "3 Teachers LSTM 3 teachers MLP 3 Teachers CNN 1 student GRU",
        "num_teachers": 9,
        "teacher_archs": ["LSTM"]*3 + ["MLP"]*3 + ["CNN"]*3,
        "student_arch": "GRU"
    },
    {
        "name": "3 teachers LSTM 1 student LSTM",
        "num_teachers": 3,
        "teacher_archs": ["LSTM"]*3,
        "student_arch": "LSTM"
    },
    {
        "name": "1 teacher LSTM 1 teacher GRU 1 Teacher CNN 1 Student MLP",
        "num_teachers": 3,
        "teacher_archs": ["LSTM", "GRU", "CNN"],
        "student_arch": "MLP"
    },
    {
        "name": "3 teachers MLP 1 student MLP",
        "num_teachers": 3,
        "teacher_archs": ["MLP"]*3,
        "student_arch": "MLP"
    },
    {
        "name": "3 teachers CNN 1 student CNN",
        "num_teachers": 3,
        "teacher_archs": ["CNN"]*3,
        "student_arch": "CNN"
    },
    {
        "name": "1 teacher MLP 1 teacher CNN 1 teacher GRU 1 teacher LSTM 1 student CNN",
        "num_teachers": 4,
        "teacher_archs": ["MLP", "CNN", "GRU", "LSTM"],
        "student_arch": "CNN"
    },
    {
        "name": "1 teacher LSTM 1 student LSTM",
        "num_teachers": 1,
        "teacher_archs": ["LSTM"],
        "student_arch": "LSTM"
    },
    {
        "name": "6 teacher LSTM 1 student LSTM",
        "num_teachers": 6,
        "teacher_archs": ["LSTM"]*6,
        "student_arch": "LSTM"
    },
    {
        "name": "6 teacher CNN 6 teacher GRU 6 teacher LSTM 1 student CNN",
        "num_teachers": 18,
        "teacher_archs": ["CNN"]*6 + ["GRU"]*6 + ["LSTM"]*6,
        "student_arch": "CNN"
    },
    {
        "name": "6 teacher MLP 6 teacher CNN 6 teacher GRU 6 teacher LSTM 1 student CNN",
        "num_teachers": 24,
        "teacher_archs": ["MLP"]*6 + ["CNN"]*6 + ["GRU"]*6 + ["LSTM"]*6,
        "student_arch": "CNN"
    },
    {
        "name": "3 teacher LSTM 3 teacher GRU 3 teacher CNN 1 student MLP",
        "num_teachers": 9,
        "teacher_archs": ["LSTM"]*3 + ["GRU"]*3 + ["CNN"]*3,
        "student_arch": "MLP"
    },
    {
        "name": "3 teacher LSTM 4 teacher MLP 3 teacher CNN 1 student GRU",
        "num_teachers": 10,
        "teacher_archs": ["LSTM"]*3 + ["MLP"]*4 + ["CNN"]*3,
        "student_arch": "GRU"
    },
    {
        "name": "10 teacher CNN 1 student CNN",
        "num_teachers": 10,
        "teacher_archs": ["CNN"]*10,
        "student_arch": "CNN"
    }
]

# Run the systematic training loop
all_results_table = []
for idx, config in enumerate(all_15_configurations, 1):
    print(f"Executing scenario {idx}/15: {config['name']}...")
    try:
        metrics = run_custom_pate_experiment(config)
        metrics["Configuration"] = config["name"]
        all_results_table.append(metrics)
    except Exception as e:
        print(f"Failed scenario {config['name']}: {str(e)}")

# Display overall consolidated statistics and save immediately to CSV
df_all_results = pd.DataFrame(all_results_table)
cols_order = ["Configuration", "dep_eps", "ind_eps", "accuracy", "precision", "recall", "f1_score", "auc", "mae", "mean_vote_margin"]
df_all_results = df_all_results[[c for c in cols_order if c in df_all_results.columns]]

# Save direct export to local file structure
df_all_results.to_csv("pate_consolidated_results.csv", index=False)
print("Saved results to 'pate_consolidated_results.csv'")

display(df_all_results)


Resolved dataset location to: /content/drive/MyDrive/data.csv
Starting PATE compliance testing...
PASS: test_dd_never_exceeds_di
All system unit tests completed successfully!
Executing scenario 1/15: 1 LSTM teacher 1 GRU student...
Executing scenario 2/15: 3 LSTM teachers 3 CNN teachers 3 MLP teachers 1 LSTM student...
Executing scenario 3/15: 3 Teachers LSTM 3 teachers MLP 3 Teachers CNN 1 student GRU...
Executing scenario 4/15: 3 teachers LSTM 1 student LSTM...
Executing scenario 5/15: 1 teacher LSTM 1 teacher GRU 1 Teacher CNN 1 Student MLP...
Executing scenario 6/15: 3 teachers MLP 1 student MLP...
Executing scenario 7/15: 3 teachers CNN 1 student CNN...
Executing scenario 8/15: 1 teacher MLP 1 teacher CNN 1 teacher GRU 1 teacher LSTM 1 student CNN...
Executing scenario 9/15: 1 teacher LSTM 1 student LSTM...
Executing scenario 10/15: 6 teacher LSTM 1 student LSTM...
Executing scenario 11/15: 6 teacher CNN 6 teacher GRU 6 teacher LSTM 1 student CNN...
Executing scenario 12/15: 6 tea

,Configuration,dep_eps,ind_eps,accuracy,precision,recall,f1_score,auc,mae,mean_vote_margin
0,1 LSTM teacher 1 GRU student,267691.159779,267691.159779,0.997850,0.997916,0.997789,0.997848,0.999855,0.249227,1.000000
1,3 LSTM teachers 3 CNN teachers 3 MLP teachers ...,84.650753,267691.159779,0.999955,0.999953,0.999956,0.999955,0.999988,0.000138,8.999845
2,3 Teachers LSTM 3 teachers MLP 3 Teachers CNN ...,84.537859,267691.159779,0.999955,0.999953,0.999956,0.999955,0.999987,0.000145,8.999806
3,3 teachers LSTM 1 student LSTM,70124.172144,267691.159779,0.999780,0.999783,0.999776,0.999780,0.999956,0.037624,3.000000
4,1 teacher LSTM 1 teacher GRU 1 Teacher CNN 1 S...,70124.172144,267691.159779,0.999955,0.999953,0.999956,0.999955,0.999971,0.031378,3.000000
5,3 teachers MLP 1 student MLP,70124.172144,267691.159779,0.999780,0.999783,0.999776,0.999780,0.999991,0.031487,3.000000
6,3 teachers CNN 1 student CNN,70133.767228,267691.159779,0.999945,0.999944,0.999946,0.999945,0.999973,0.027652,2.999903
7,1 teacher MLP 1 teacher CNN 1 teacher GRU 1 te...,24230.854030,267691.159779,0.999948,0.999947,0.999949,0.999948,0.999998,0.008202,4.000000
8,1 teacher LSTM 1 student LSTM,267691.159779,267691.159779,0.999369,0.999360,0.999376,0.999368,0.999848,0.239236,1.000000
9,6 teacher LSTM 1 student LSTM,2572.109945,267691.159779,0.999955,0.999953,0.999956,0.999955,0.999975,0.001666,6.000000
